# Gradient-flow W_temp plots

This notebook reads the saved output folder from `analyze_gradient_flow_wtemp.py`. Most plots use stored plot-ready summaries; the optional tau-int cell rereads one selected raw `W_temp` series only.

In [1]:
from pathlib import Path
import json
import math

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

try:
    from finalized_analysis_helpers import _ensure_plotly_browser_path
    _ensure_plotly_browser_path()
except Exception:
    pass

pio.templates.default = "plotly_white"

# Point this at either one group folder containing summary.json or at the
# parent gradient_flow_wtemp_analysis folder containing several group folders.
ANALYSIS_ROOT = Path("../data/gradient_flow_wtemp_analysis/beta_2p4__L0_24__epsilon1_0__dt_0p01__b6c467be")
ANALYSIS_INDEX = 0

# Main Wilson-loop plot controls.
SELECTED_FLOW_TIME = None      # None gives an interactive dropdown; e.g. 0.0 saves/plots only that flow time.
SELECTED_L_VALUES = None       # None keeps all L values; e.g. [4, 6, 8]
SELECTED_T_VALUES = None       # None keeps all T values; e.g. range(2, 16)
SHOW_ERROR_BARS = True
LOG_Y = False
FIGURE_WIDTH = 950
FIGURE_HEIGHT = 600

# Thermalization preview controls. Use tuples like (0.0, 4, 4) for t/a^2, L, T.
THERMALIZATION_KEY = None
THERMALIZATION_MAX_SERIES_PER_KEY = None
THERMALIZATION_VISIBLE_RUNS = None          # None keeps all run labels; e.g. ["run_0", "run_1"]
THERMALIZATION_X_RANGE = None              # None autoscales; e.g. (0, 4500)
THERMALIZATION_RAW_Y_RANGE = None          # y range for the W(L,T) plot; e.g. (0.628, 0.675)
THERMALIZATION_CUT = None                  # None uses the suggested cut; e.g. 1500
THERMALIZATION_SHOW_SELECTOR = True        # Show the in-plot selector dropdown.

# Bootstrap block-size plot controls. Keys are tuples like (0.0, 4, 4) for t/a^2, L, T.
BLOCK_SIZE_KEYS = None                 # None keeps all curves; e.g. [(0.0, 4, 4)]
BLOCK_SIZE_X_RANGE = None              # None autoscales; e.g. (0, 1200)
BLOCK_SIZE_Y_RANGE = None              # None autoscales; e.g. (0.0, 0.002)
BLOCK_SIZE_DEFAULT = None              # None uses the saved default block size.

# Integrated autocorrelation plot controls for one W(R,T) at one flow time.
TAU_FLOW_TIME = 1.0
TAU_WILSON_LOOP = (4, 4)
TAU_INPUT_INDEX = 1                 # Select one raw file from summary["input_paths"]: 1 is run folder 26.
TAU_INPUT_PATH = None               # Optional explicit path; overrides TAU_INPUT_INDEX.
TAU_THERMALIZATION_CUT = 1000
TAU_MAX_LAG = 300
TAU_WINDOW_M = 150                 # None uses the automatic window rule.
TAU_WINDOW_C = 5.0
TAU_ALL_FILES_OPACITY = 0.35
SAVE_TAU_INT_PDF = True
TAU_INT_PDF_PATH = None  # None writes <analysis_dir>/tau_int_single_file.pdf
SAVE_TAU_AUTOCORRELATION_PDF = True
TAU_AUTOCORRELATION_PDF_PATH = None  # None writes <analysis_dir>/tau_autocorrelation_single_file.pdf

# Final PDF export controls. Choices: "wilson", "thermalization", "block_size", "tau_int", "tau_autocorrelation", "tau_int_all", "tau_avg_autocorrelation".
PDF_FIGURE = "wilson"
PDF_PATH = None                # None writes <analysis_dir>/<PDF_FIGURE>_plot.pdf


In [2]:
def load_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def find_analysis_dirs(root):
    root = Path(root).expanduser()
    if (root / "summary.json").is_file():
        return [root.resolve()]
    if not root.exists():
        raise FileNotFoundError(f"Analysis folder does not exist: {root}")
    return [path.parent.resolve() for path in sorted(root.rglob("summary.json"))]


analysis_dirs = find_analysis_dirs(ANALYSIS_ROOT)
if not analysis_dirs:
    raise FileNotFoundError(f"No summary.json files found below {ANALYSIS_ROOT}")
if ANALYSIS_INDEX < 0 or ANALYSIS_INDEX >= len(analysis_dirs):
    raise IndexError(f"ANALYSIS_INDEX must be between 0 and {len(analysis_dirs) - 1}")

print("Available analysis folders:")
for index, path in enumerate(analysis_dirs):
    marker = "*" if index == ANALYSIS_INDEX else " "
    print(f"{marker} [{index}] {path}")

analysis_dir = analysis_dirs[ANALYSIS_INDEX]
summary = load_json(analysis_dir / "summary.json")
records = [dict(row) for row in summary.get("records", [])]
if not records:
    raise ValueError(f"No Wilson-loop records found in {analysis_dir / 'summary.json'}")

flow_times = sorted({float(row["t_over_a2"]) for row in records})
print(f"\nSelected: {analysis_dir}")
print(f"Wilson loops: {len(records)}")
print(f"flow times: {[f'{flow:g}' for flow in flow_times]}")
print(f"thermalization cuts: {summary.get('thermalization_by_input', {})}")
print(f"block size: {summary.get('block_size')}, n_bootstrap: {summary.get('n_bootstrap')}")

figure_registry = {}


Available analysis folders:
* [0] /home/pietl/masterarbeit/data/gradient_flow_wtemp_analysis/beta_2p4__L0_24__epsilon1_0__dt_0p01__b6c467be

Selected: /home/pietl/masterarbeit/data/gradient_flow_wtemp_analysis/beta_2p4__L0_24__epsilon1_0__dt_0p01__b6c467be
Wilson loops: 980
flow times: ['0', '0.25', '0.5', '0.75', '1']
thermalization cuts: {'/home/pietl/masterarbeit/data/20260524/25/gradient_flow_wtemp.dat': 800, '/home/pietl/masterarbeit/data/20260524/26/gradient_flow_wtemp.dat': 800, '/home/pietl/masterarbeit/data/20260524/27/gradient_flow_wtemp.dat': 800, '/home/pietl/masterarbeit/data/20260524/28/gradient_flow_wtemp.dat': 800, '/home/pietl/masterarbeit/data/20260524/29/gradient_flow_wtemp.dat': 800, '/home/pietl/masterarbeit/data/20260524/30/gradient_flow_wtemp.dat': 800, '/home/pietl/masterarbeit/data/20260524/31/gradient_flow_wtemp.dat': 800, '/home/pietl/masterarbeit/data/20260524/32/gradient_flow_wtemp.dat': 800, '/home/pietl/masterarbeit/data/20260531/1/gradient_flow_wtemp.dat

In [3]:
def _int_selection(values):
    if values is None:
        return None
    return {int(value) for value in values}


def _closest_flow(requested, available):
    if requested is None:
        return None
    target = float(requested)
    return min(available, key=lambda value: abs(float(value) - target))


def _same_flow(left, right):
    return math.isclose(float(left), float(right), rel_tol=1e-12, abs_tol=1e-12)


def _filtered_wilson_records(source_records, flow_time=None, l_values=None, t_values=None):
    l_set = _int_selection(l_values)
    t_set = _int_selection(t_values)
    available_flows = sorted({float(row["t_over_a2"]) for row in source_records})
    selected_flow = _closest_flow(flow_time, available_flows)
    filtered = []
    for row in source_records:
        if selected_flow is not None and not _same_flow(row["t_over_a2"], selected_flow):
            continue
        if l_set is not None and int(row["L"]) not in l_set:
            continue
        if t_set is not None and int(row["T"]) not in t_set:
            continue
        filtered.append(row)
    return filtered, selected_flow


def build_wilson_loop_figure(
    source_records,
    *,
    flow_time=None,
    l_values=None,
    t_values=None,
    show_error_bars=True,
    log_y=False,
    width=950,
    height=600,
):
    filtered, selected_flow = _filtered_wilson_records(
        source_records,
        flow_time=flow_time,
        l_values=l_values,
        t_values=t_values,
    )
    if not filtered:
        raise ValueError("No records remain after the selected plot filters.")

    flows = sorted({float(row["t_over_a2"]) for row in filtered})
    fig = go.Figure()
    trace_indices_by_flow = {}

    for flow_index, flow in enumerate(flows):
        visible = flow_index == 0
        trace_indices_by_flow[flow] = []
        rows_for_flow = [row for row in filtered if _same_flow(row["t_over_a2"], flow)]
        for l_size in sorted({int(row["L"]) for row in rows_for_flow}):
            rows = sorted(
                [row for row in rows_for_flow if int(row["L"]) == l_size],
                key=lambda row: int(row["T"]),
            )
            trace_indices_by_flow[flow].append(len(fig.data))
            trace = {
                "x": [int(row["T"]) for row in rows],
                "y": [float(row["mean"]) for row in rows],
                "mode": "lines+markers",
                "name": f"L={l_size}",
                "legendgroup": f"flow_{flow:g}",
                "visible": visible,
                "customdata": [
                    [
                        int(row["L"]),
                        int(row["T"]),
                        int(row["n_measurements"]),
                        float(row.get("bootstrap_error") or row.get("error") or 0.0),
                    ]
                    for row in rows
                ],
                "hovertemplate": (
                    "L=%{customdata[0]}<br>"
                    "T=%{customdata[1]}<br>"
                    "n=%{customdata[2]}<br>"
                    "mean=%{y:.8g}<br>"
                    "bootstrap error=%{customdata[3]:.3g}"
                    "<extra>t/a^2 " + f"{flow:g}" + "</extra>"
                ),
            }
            if show_error_bars:
                trace["error_y"] = {
                    "type": "data",
                    "array": [float(row.get("bootstrap_error") or row.get("error") or 0.0) for row in rows],
                    "visible": True,
                }
            fig.add_trace(go.Scatter(**trace))

    if len(flows) > 1 and selected_flow is None:
        buttons = []
        for flow in flows:
            visible = [False] * len(fig.data)
            for trace_index in trace_indices_by_flow[flow]:
                visible[trace_index] = True
            buttons.append(
                {
                    "label": f"t/a^2={flow:g}",
                    "method": "update",
                    "args": [
                        {"visible": visible},
                        {"title": f"Gradient-flow Wilson loops at t/a^2={flow:g}"},
                    ],
                }
            )
        fig.update_layout(
            updatemenus=[
                {
                    "buttons": buttons,
                    "direction": "down",
                    "x": 1.0,
                    "xanchor": "right",
                    "y": 1.15,
                    "yanchor": "top",
                }
            ]
        )

    title_flow = flows[0]
    fig.update_layout(
        title=f"Gradient-flow Wilson loops at t/a^2={title_flow:g}",
        width=width,
        height=height,
        xaxis_title="T",
        yaxis_title="W(L,T)",
        legend_title="Spatial size",
        margin={"l": 48, "r": 12, "t": 12, "b": 46},
    )
    fig.update_yaxes(type="log" if log_y else "linear")
    return fig


fig_wilson = build_wilson_loop_figure(
    records,
    flow_time=SELECTED_FLOW_TIME,
    l_values=SELECTED_L_VALUES,
    t_values=SELECTED_T_VALUES,
    show_error_bars=SHOW_ERROR_BARS,
    log_y=LOG_Y,
    width=FIGURE_WIDTH,
    height=FIGURE_HEIGHT,
)
figure_registry["wilson"] = fig_wilson
print("Wilson-loop plot is ready. Use build_wilson_loop_figure(...) or the PDF export cell if needed.")


Wilson-loop plot is ready. Use build_wilson_loop_figure(...) or the PDF export cell if needed.


In [4]:
def _preview_key(row):
    return (float(row["t_over_a2"]), int(row["L"]), int(row["T"]))


def _format_preview_key(key):
    return f"t={key[0]:g}, L={key[1]}, T={key[2]}"


def _wilson_loop_label(l_size, t_size):
    return f"W({int(l_size)},{int(t_size)})"


def _choose_preview_key(records_for_preview, requested_key):
    keys = sorted({_preview_key(row) for row in records_for_preview})
    if not keys:
        raise ValueError("No thermalization preview records are available.")
    if requested_key is None:
        return keys[0]
    target = (float(requested_key[0]), int(requested_key[1]), int(requested_key[2]))
    for key in keys:
        if _same_flow(key[0], target[0]) and key[1:] == target[1:]:
            return key
    raise ValueError(f"Requested THERMALIZATION_KEY {requested_key!r} is not available.")


def _axis_range(value, *, name):
    if value is None:
        return None
    if len(value) != 2:
        raise ValueError(f"{name} must be a pair: (start, endpoint).")
    start, endpoint = value
    if start is None or endpoint is None:
        return None
    start = float(start)
    endpoint = float(endpoint)
    if start >= endpoint:
        raise ValueError(f"{name} start must be smaller than endpoint.")
    return [start, endpoint]


def _visible_run_label_set(visible_run_labels):
    if visible_run_labels is None:
        return None
    return {str(label) for label in visible_run_labels}


def _run_label_sort_key(label):
    label = str(label)
    try:
        return (0, int(label))
    except ValueError:
        return (1, label)


def _run_labels_for_key(preview_records, key):
    return sorted(
        {str(row["run_label"]) for row in preview_records if _preview_key(row) == key},
        key=_run_label_sort_key,
    )


def _thermalization_cut(payload, requested_cut):
    if requested_cut is None:
        cut = payload.get("suggested_thermalization_cut")
    else:
        cut = requested_cut
    if cut is None:
        return None
    return float(cut)


def build_thermalization_figure(
    payload,
    *,
    selected_key=None,
    max_series_per_key=None,
    visible_run_labels=None,
    x_range=None,
    raw_y_range=None,
    cut=None,
    dropdown=True,
    width=950,
    height=560,
):
    preview_records = [dict(row) for row in payload.get("records", [])]
    initial_key = _choose_preview_key(preview_records, selected_key)
    visible_labels = _visible_run_label_set(visible_run_labels)
    if visible_labels is not None:
        preview_records = [row for row in preview_records if str(row["run_label"]) in visible_labels]
    if not preview_records:
        raise ValueError("No thermalization preview curves remain after THERMALIZATION_VISIBLE_RUNS.")

    keys = sorted({_preview_key(row) for row in preview_records})
    if initial_key not in keys:
        raise ValueError(
            "No curves remain for the selected thermalization key after THERMALIZATION_VISIBLE_RUNS."
        )
    if not dropdown:
        keys = [initial_key]

    fig = go.Figure()
    trace_indices_by_key = {}
    for key in keys:
        visible = key == initial_key
        trace_indices_by_key[key] = []
        rows_for_key = [row for row in preview_records if _preview_key(row) == key]
        if max_series_per_key is not None:
            rows_for_key = rows_for_key[: int(max_series_per_key)]
        for row in rows_for_key:
            x_values = [float(value) for value in row["configuration_id"]]
            y_values = [float(value) for value in row["W_temp"]]
            name = str(row["run_label"])
            trace_indices_by_key[key].append(len(fig.data))
            fig.add_trace(
                go.Scatter(
                    x=x_values,
                    y=y_values,
                    mode="markers",
                    name=name,
                    visible=visible,
                    opacity=0.55,
                    legendgroup=name,
                )
            )

    cut = _thermalization_cut(payload, cut)
    shapes = []
    annotations = []
    if cut is not None:
        shapes.append(
            {
                "type": "line",
                "xref": "x",
                "yref": "paper",
                "x0": float(cut),
                "x1": float(cut),
                "y0": 0,
                "y1": 1,
                "line": {"color": "red", "width": 1.5, "dash": "dash"},
            }
        )
    initial_w_label = _wilson_loop_label(initial_key[1], initial_key[2])

    if dropdown and len(keys) > 1:
        buttons = []
        for key in keys:
            visible = [False] * len(fig.data)
            for trace_index in trace_indices_by_key[key]:
                visible[trace_index] = True
            buttons.append(
                {
                    "label": _format_preview_key(key),
                    "method": "update",
                    "args": [
                        {"visible": visible},
                    ],
                }
            )
        fig.update_layout(
            updatemenus=[
                {
                    "buttons": buttons,
                    "active": keys.index(initial_key),
                    "direction": "down",
                    "x": 1.0,
                    "xanchor": "right",
                    "y": 1.04,
                    "yanchor": "top",
                }
            ]
        )

    fig.update_layout(
        title=None,
        width=width,
        height=height,
        xaxis_title="configuration index",
        yaxis_title=initial_w_label,
        margin={"l": 52, "r": 12, "t": 18, "b": 46},
        shapes=shapes,
        annotations=annotations,
    )

    x_range = _axis_range(x_range, name="THERMALIZATION_X_RANGE")
    raw_y_range = _axis_range(raw_y_range, name="THERMALIZATION_RAW_Y_RANGE")
    if x_range is not None:
        fig.update_xaxes(range=x_range)
    if raw_y_range is not None:
        fig.update_yaxes(range=raw_y_range)
    return fig


thermalization_payload = None
thermalization_path = analysis_dir / "thermalization_preview_data.json"
if thermalization_path.exists():
    thermalization_payload = load_json(thermalization_path)
    fig_thermalization = build_thermalization_figure(
        thermalization_payload,
        selected_key=THERMALIZATION_KEY,
        max_series_per_key=THERMALIZATION_MAX_SERIES_PER_KEY,
        visible_run_labels=THERMALIZATION_VISIBLE_RUNS,
        x_range=THERMALIZATION_X_RANGE,
        raw_y_range=THERMALIZATION_RAW_Y_RANGE,
        cut=THERMALIZATION_CUT,
        dropdown=THERMALIZATION_SHOW_SELECTOR,
        width=FIGURE_WIDTH,
    )
    figure_registry["thermalization"] = fig_thermalization
    print("Thermalization plot is ready. Use the editor cell below to display or save the edited plot.")
else:
    print(f"No thermalization preview data found at {thermalization_path}")
    print("Rerun analyze_gradient_flow_wtemp.py without an explicit THERM argument to create it.")


Thermalization plot is ready. Use the editor cell below to display or save the edited plot.


In [5]:
# Optional interactive editor for the thermalization plot.
# Run this cell, adjust the controls, then press "Save PDF".
try:
    import ipywidgets as widgets
    from IPython.display import clear_output, display
except Exception as exc:
    print(f"Interactive editor unavailable: {exc}")
else:
    clear_output(wait=True)
    if thermalization_payload is None:
        print("No thermalization preview data is available for the editor.")
    else:
        _preview_records_for_editor = [dict(row) for row in thermalization_payload.get("records", [])]
        _editor_keys = sorted({_preview_key(row) for row in _preview_records_for_editor})
        _initial_editor_key = _choose_preview_key(_preview_records_for_editor, THERMALIZATION_KEY)

        selector = widgets.Dropdown(
            options=[(_format_preview_key(key), key) for key in _editor_keys],
            value=_initial_editor_key,
            description="Selector",
            layout=widgets.Layout(width="260px"),
        )
        curve_select = widgets.SelectMultiple(
            description="Curves",
            rows=8,
            layout=widgets.Layout(width="260px"),
        )
        show_selector = widgets.Checkbox(value=THERMALIZATION_SHOW_SELECTOR, description="Show plot selector")
        x_start = widgets.Text(value="", description="x start", layout=widgets.Layout(width="155px"))
        x_end = widgets.Text(value="", description="x end", layout=widgets.Layout(width="155px"))
        raw_y_start = widgets.Text(value="", description="raw y start", layout=widgets.Layout(width="155px"))
        raw_y_end = widgets.Text(value="", description="raw y end", layout=widgets.Layout(width="155px"))
        cut_value = widgets.Text(value="", description="cut", layout=widgets.Layout(width="155px"))
        def _default_pdf_path(key):
            return analysis_dir / f"thermalization_W{key[1]}{key[2]}.pdf"

        pdf_output_path = widgets.Text(
            value=str(_default_pdf_path(_initial_editor_key)),
            description="PDF path",
            layout=widgets.Layout(width="520px"),
        )
        last_default_pdf_path = {"value": str(_default_pdf_path(_initial_editor_key))}
        redraw_button = widgets.Button(description="Update plot", button_style="primary")
        save_button = widgets.Button(description="Save PDF")
        status_output = widgets.Output()
        preview_image = widgets.Image(
            value=b"",
            format="png",
            layout=widgets.Layout(width=f"{FIGURE_WIDTH}px"),
        )

        def _set_range_texts(range_value, start_widget, end_widget):
            parsed = _axis_range(range_value, name="range")
            if parsed is None:
                return
            start_widget.value = f"{parsed[0]:g}"
            end_widget.value = f"{parsed[1]:g}"

        def _parse_optional_float(text, label):
            value = text.strip()
            if not value:
                return None
            try:
                return float(value)
            except ValueError as exc:
                raise ValueError(f"{label} must be a number or blank.") from exc

        def _range_from_widgets(start_widget, end_widget, label):
            start = _parse_optional_float(start_widget.value, f"{label} start")
            endpoint = _parse_optional_float(end_widget.value, f"{label} endpoint")
            if start is None and endpoint is None:
                return None
            return _axis_range((start, endpoint), name=label)

        def _cut_from_widget():
            return _parse_optional_float(cut_value.value, "cut")

        def _refresh_curves(*_args):
            labels = _run_labels_for_key(_preview_records_for_editor, selector.value)
            curve_select.options = labels
            configured = _visible_run_label_set(THERMALIZATION_VISIBLE_RUNS)
            if configured is None:
                curve_select.value = tuple(labels)
            else:
                curve_select.value = tuple(label for label in labels if label in configured)
            new_default_path = str(_default_pdf_path(selector.value))
            if not pdf_output_path.value.strip() or pdf_output_path.value == last_default_pdf_path["value"]:
                pdf_output_path.value = new_default_path
            last_default_pdf_path["value"] = new_default_path

        def _current_editor_figure():
            selected_curves = list(curve_select.value)
            if not selected_curves:
                raise ValueError("Select at least one curve.")
            return build_thermalization_figure(
                thermalization_payload,
                selected_key=selector.value,
                max_series_per_key=THERMALIZATION_MAX_SERIES_PER_KEY,
                visible_run_labels=selected_curves,
                x_range=_range_from_widgets(x_start, x_end, "x range"),
                raw_y_range=_range_from_widgets(raw_y_start, raw_y_end, "raw y range"),
                cut=_cut_from_widget(),
                dropdown=show_selector.value,
                width=FIGURE_WIDTH,
                height=FIGURE_HEIGHT,
            )

        def _redraw(_button=None):
            status_output.clear_output(wait=True)
            preview_image.value = b""
            try:
                fig = _current_editor_figure()
                figure_registry["thermalization_editor"] = fig
                preview_image.value = fig.to_image(format="png", scale=2)
            except Exception as exc:
                with status_output:
                    print(f"Could not update plot: {exc}")

        def _save_pdf(_button=None):
            status_output.clear_output(wait=True)
            try:
                fig = _current_editor_figure()
                path = Path(pdf_output_path.value).expanduser()
                path.parent.mkdir(parents=True, exist_ok=True)
                fig.write_image(str(path))
                figure_registry["thermalization_editor"] = fig
                with status_output:
                    print(f"Saved PDF: {path}")
            except Exception as exc:
                with status_output:
                    print(f"Could not save PDF: {exc}")

        _set_range_texts(THERMALIZATION_X_RANGE, x_start, x_end)
        _set_range_texts(THERMALIZATION_RAW_Y_RANGE, raw_y_start, raw_y_end)
        cut_value.value = "" if THERMALIZATION_CUT is None else f"{float(THERMALIZATION_CUT):g}"
        _refresh_curves()
        selector.observe(_refresh_curves, names="value")
        redraw_button.on_click(_redraw)
        save_button.on_click(_save_pdf)

        display(
            widgets.VBox(
                [
                    widgets.HBox([selector, show_selector]),
                    curve_select,
                    widgets.HBox([x_start, x_end]),
                    widgets.HBox([raw_y_start, raw_y_end]),
                    widgets.HBox([cut_value]),
                    widgets.HBox([pdf_output_path, redraw_button, save_button]),
                    status_output,
                    preview_image,
                ]
            )
        )
        print("Set the controls, then press Update plot or Save PDF.")


Set the controls, then press Update plot or Save PDF.


In [6]:
def _block_size_key(row):
    return (float(row["t_over_a2"]), int(row["L"]), int(row["T"]))


def _format_block_size_key(key):
    return f"t={key[0]:g}, L={key[1]}, T={key[2]}"


def _normalize_block_size_keys(requested_keys):
    if requested_keys is None:
        return None
    if len(requested_keys) == 3 and not isinstance(requested_keys[0], (list, tuple)):
        requested_keys = [requested_keys]
    return {(float(key[0]), int(key[1]), int(key[2])) for key in requested_keys}


def _default_block_size(payload, requested_block_size):
    if requested_block_size is None:
        block_size = payload.get("default_block_size")
    else:
        block_size = requested_block_size
    if block_size is None:
        return None
    return float(block_size)


def _block_size_y_label(keys):
    loop_keys = {(int(key[1]), int(key[2])) for key in keys}
    if len(loop_keys) == 1:
        l_size, t_size = next(iter(loop_keys))
        return f"bootstrap error of {_wilson_loop_label(l_size, t_size)}"
    return "bootstrap error"


def build_block_size_figure(
    payload,
    *,
    selected_keys=None,
    x_range=None,
    y_range=None,
    default_block_size=None,
    width=950,
    height=560,
):
    scan_records = [dict(row) for row in payload.get("records", [])]
    if not scan_records:
        raise ValueError("No block-size scan records are available.")

    requested_keys = _normalize_block_size_keys(selected_keys)
    if requested_keys is not None:
        scan_records = [row for row in scan_records if _block_size_key(row) in requested_keys]
    if not scan_records:
        raise ValueError("No block-size scan records remain after the selected filters.")

    keys = sorted({_block_size_key(row) for row in scan_records})
    fig = go.Figure()
    for flow, l_size, t_size in keys:
        rows = sorted(
            [row for row in scan_records if _block_size_key(row) == (flow, l_size, t_size)],
            key=lambda row: int(row["block_size"]),
        )
        fig.add_trace(
            go.Scatter(
                x=[int(row["block_size"]) for row in rows],
                y=[float(row["error"]) for row in rows],
                mode="markers",
                name=f"t={flow:g}, L={l_size}, T={t_size}",
            )
        )

    block_size = _default_block_size(payload, default_block_size)
    shapes = []
    if block_size is not None:
        shapes.append(
            {
                "type": "line",
                "xref": "x",
                "yref": "paper",
                "x0": float(block_size),
                "x1": float(block_size),
                "y0": 0,
                "y1": 1,
                "line": {"color": "red", "width": 1.5, "dash": "dash"},
            }
        )

    fig.update_layout(
        title=None,
        width=width,
        height=height,
        xaxis_title="block size",
        yaxis_title=_block_size_y_label(keys),
        legend_title="Loop",
        margin={"l": 52, "r": 12, "t": 18, "b": 46},
        shapes=shapes,
    )

    x_range = _axis_range(x_range, name="BLOCK_SIZE_X_RANGE")
    y_range = _axis_range(y_range, name="BLOCK_SIZE_Y_RANGE")
    if x_range is not None:
        fig.update_xaxes(range=x_range)
    fig.update_yaxes(exponentformat="power", showexponent="all")
    if y_range is not None:
        fig.update_yaxes(range=y_range)
    return fig


block_size_payload = None
block_size_path = analysis_dir / "bootstrap_block_size_scan.json"
if block_size_path.exists():
    block_size_payload = load_json(block_size_path)
    fig_block_size = build_block_size_figure(
        block_size_payload,
        selected_keys=BLOCK_SIZE_KEYS,
        x_range=BLOCK_SIZE_X_RANGE,
        y_range=BLOCK_SIZE_Y_RANGE,
        default_block_size=BLOCK_SIZE_DEFAULT,
        width=FIGURE_WIDTH,
    )
    figure_registry["block_size"] = fig_block_size
    print("Block-size plot is ready. Use the bootstrap editor cell below to display or save the edited plot.")
else:
    print(f"No block-size scan data found at {block_size_path}")
    print("This file is created when analyze_gradient_flow_wtemp.py prompts for BLOCK.")


Block-size plot is ready. Use the bootstrap editor cell below to display or save the edited plot.


In [7]:
# Optional interactive editor for the bootstrap block-size plot.
# Run this cell, adjust the controls, then press "Save PDF".
try:
    import ipywidgets as widgets
    from IPython.display import clear_output, display
except Exception as exc:
    print(f"Interactive editor unavailable: {exc}")
else:
    clear_output(wait=True)
    if block_size_payload is None:
        print("No bootstrap block-size scan data is available for the editor.")
    else:
        _block_records_for_editor = [dict(row) for row in block_size_payload.get("records", [])]
        _block_keys = sorted({_block_size_key(row) for row in _block_records_for_editor})

        block_curve_select = widgets.SelectMultiple(
            options=[(_format_block_size_key(key), key) for key in _block_keys],
            value=tuple(_block_keys if BLOCK_SIZE_KEYS is None else sorted(_normalize_block_size_keys(BLOCK_SIZE_KEYS))),
            description="Curves",
            rows=8,
            layout=widgets.Layout(width="300px"),
        )
        block_x_start = widgets.Text(value="", description="x start", layout=widgets.Layout(width="155px"))
        block_x_end = widgets.Text(value="", description="x end", layout=widgets.Layout(width="155px"))
        block_y_start = widgets.Text(value="", description="y start", layout=widgets.Layout(width="155px"))
        block_y_end = widgets.Text(value="", description="y end", layout=widgets.Layout(width="155px"))
        block_default = widgets.Text(value="", description="default", layout=widgets.Layout(width="155px"))

        def _default_block_pdf_path(keys):
            loop_keys = {(int(key[1]), int(key[2])) for key in keys}
            if len(loop_keys) == 1:
                l_size, t_size = next(iter(loop_keys))
                return analysis_dir / f"bootstrap_block_size_W{l_size}{t_size}.pdf"
            return analysis_dir / "bootstrap_block_size.pdf"

        block_pdf_output_path = widgets.Text(
            value=str(_default_block_pdf_path(block_curve_select.value)),
            description="PDF path",
            layout=widgets.Layout(width="520px"),
        )
        last_block_default_pdf_path = {"value": block_pdf_output_path.value}
        block_redraw_button = widgets.Button(description="Update plot", button_style="primary")
        block_save_button = widgets.Button(description="Save PDF")
        block_status_output = widgets.Output()
        block_preview_image = widgets.Image(
            value=b"",
            format="png",
            layout=widgets.Layout(width=f"{FIGURE_WIDTH}px"),
        )

        def _parse_optional_float(text, label):
            value = text.strip()
            if not value:
                return None
            try:
                return float(value)
            except ValueError as exc:
                raise ValueError(f"{label} must be a number or blank.") from exc

        def _range_from_widgets(start_widget, end_widget, label):
            start = _parse_optional_float(start_widget.value, f"{label} start")
            endpoint = _parse_optional_float(end_widget.value, f"{label} endpoint")
            if start is None and endpoint is None:
                return None
            return _axis_range((start, endpoint), name=label)

        def _set_range_texts(range_value, start_widget, end_widget):
            parsed = _axis_range(range_value, name="range")
            if parsed is None:
                return
            start_widget.value = f"{parsed[0]:g}"
            end_widget.value = f"{parsed[1]:g}"

        def _selected_block_keys():
            keys = list(block_curve_select.value)
            if not keys:
                raise ValueError("Select at least one curve.")
            return keys

        def _refresh_block_pdf_path(*_args):
            keys = list(block_curve_select.value)
            if not keys:
                return
            new_default_path = str(_default_block_pdf_path(keys))
            if not block_pdf_output_path.value.strip() or block_pdf_output_path.value == last_block_default_pdf_path["value"]:
                block_pdf_output_path.value = new_default_path
            last_block_default_pdf_path["value"] = new_default_path

        def _current_block_figure():
            return build_block_size_figure(
                block_size_payload,
                selected_keys=_selected_block_keys(),
                x_range=_range_from_widgets(block_x_start, block_x_end, "block-size x range"),
                y_range=_range_from_widgets(block_y_start, block_y_end, "block-size y range"),
                default_block_size=_parse_optional_float(block_default.value, "default block size"),
                width=FIGURE_WIDTH,
                height=FIGURE_HEIGHT,
            )

        def _redraw_block(_button=None):
            block_status_output.clear_output(wait=True)
            block_preview_image.value = b""
            try:
                fig = _current_block_figure()
                figure_registry["block_size_editor"] = fig
                block_preview_image.value = fig.to_image(format="png", scale=2)
            except Exception as exc:
                with block_status_output:
                    print(f"Could not update plot: {exc}")

        def _save_block_pdf(_button=None):
            block_status_output.clear_output(wait=True)
            try:
                fig = _current_block_figure()
                path = Path(block_pdf_output_path.value).expanduser()
                path.parent.mkdir(parents=True, exist_ok=True)
                fig.write_image(str(path))
                figure_registry["block_size_editor"] = fig
                with block_status_output:
                    print(f"Saved PDF: {path}")
            except Exception as exc:
                with block_status_output:
                    print(f"Could not save PDF: {exc}")

        _set_range_texts(BLOCK_SIZE_X_RANGE, block_x_start, block_x_end)
        _set_range_texts(BLOCK_SIZE_Y_RANGE, block_y_start, block_y_end)
        block_default.value = "" if BLOCK_SIZE_DEFAULT is None else f"{float(BLOCK_SIZE_DEFAULT):g}"
        block_curve_select.observe(_refresh_block_pdf_path, names="value")
        block_redraw_button.on_click(_redraw_block)
        block_save_button.on_click(_save_block_pdf)

        display(
            widgets.VBox(
                [
                    block_curve_select,
                    widgets.HBox([block_x_start, block_x_end]),
                    widgets.HBox([block_y_start, block_y_end]),
                    widgets.HBox([block_default]),
                    widgets.HBox([block_pdf_output_path, block_redraw_button, block_save_button]),
                    block_status_output,
                    block_preview_image,
                ]
            )
        )
        print("Set the bootstrap controls, then press Update plot or Save PDF.")


Set the bootstrap controls, then press Update plot or Save PDF.


## Integrated autocorrelation time

This cell calculates and plots `tau_int(M)` for one selected `W(R,T)` and one flow time from one selected raw `W_temp` file after the chosen thermalization cut.

In [8]:
def _split_wtemp_fields(stripped):
    return stripped.split(",") if "," in stripped else stripped.split()


def _is_wtemp_data_line(stripped):
    if not stripped or stripped.startswith("#"):
        return False
    fields = _split_wtemp_fields(stripped)
    if not fields:
        return False
    try:
        float(fields[0])
    except ValueError:
        return False
    return True


def _iter_wtemp_data_lines(path):
    with Path(path).open("r", encoding="utf-8-sig") as handle:
        for line in handle:
            stripped = line.strip()
            if not stripped or stripped.startswith("#"):
                continue
            first = stripped[0]
            if first.isdigit() or first in "+-.":
                yield stripped


def _parse_wtemp_data_row(stripped):
    fields = _split_wtemp_fields(stripped)
    if len(fields) != 5:
        raise ValueError(f"Expected 5 columns in gradient-flow W_temp row, got {len(fields)}: {stripped[:120]}")
    return (
        float(fields[0]),
        float(fields[1]),
        int(float(fields[2])),
        int(float(fields[3])),
        float(fields[4]),
    )


def _wtemp_key_position(path, flow_time, l_size, t_size):
    first_conf_id = None
    key_order = []
    for stripped in _iter_wtemp_data_lines(path):
        conf_id, row_flow, row_l, row_t, _ = _parse_wtemp_data_row(stripped)
        if first_conf_id is None:
            first_conf_id = conf_id
        elif conf_id != first_conf_id:
            break
        key_order.append((row_flow, row_l, row_t))

    if not key_order:
        raise ValueError(f"No W_temp data rows found in {path}")

    for index, (row_flow, row_l, row_t) in enumerate(key_order):
        if _same_flow(row_flow, flow_time) and row_l == int(l_size) and row_t == int(t_size):
            return len(key_order), index

    raise ValueError(f"{path}: no W({int(l_size)},{int(t_size)}) row at t/a^2={float(flow_time):g}")


def _load_wtemp_series_from_file(path, *, flow_time, l_size, t_size, thermalization_cut):
    rows_per_conf, target_row = _wtemp_key_position(path, flow_time, l_size, t_size)
    values = []
    data_row_index = 0
    for stripped in _iter_wtemp_data_lines(path):
        if data_row_index % rows_per_conf == target_row:
            conf_id, _, _, _, w_temp = _parse_wtemp_data_row(stripped)
            if conf_id >= float(thermalization_cut):
                values.append(w_temp)
        data_row_index += 1
    return np.asarray(values, dtype=np.float64)


def _summary_input_paths(summary):
    paths = [Path(path).expanduser() for path in summary.get("input_paths", [])]
    if not paths:
        raise ValueError("summary.json does not contain input_paths; rerun analyze_gradient_flow_wtemp.py with saved input paths.")
    missing = [path for path in paths if not path.exists()]
    if missing:
        preview = "\n".join(str(path) for path in missing[:5])
        raise FileNotFoundError(f"{len(missing)} raw W_temp file(s) are missing. First missing paths:\n{preview}")
    return paths


def select_tau_input_path(summary, *, input_path=None, input_index=0):
    paths = _summary_input_paths(summary)
    if input_path is not None:
        path = Path(input_path).expanduser()
        if not path.exists():
            raise FileNotFoundError(f"Selected TAU_INPUT_PATH does not exist: {path}")
        return path, None, len(paths)

    index = int(input_index)
    if index < 0 or index >= len(paths):
        raise IndexError(f"TAU_INPUT_INDEX must be between 0 and {len(paths) - 1}")
    return paths[index], index, len(paths)


def load_wtemp_series_for_key(summary, *, flow_time, l_size, t_size, thermalization_cut, input_path=None, input_index=0):
    path, selected_index, n_available = select_tau_input_path(
        summary,
        input_path=input_path,
        input_index=input_index,
    )
    selection_label = f"input_paths[{selected_index}]" if selected_index is not None else "TAU_INPUT_PATH"
    print(
        f"Loading W({int(l_size)},{int(t_size)}) at t/a^2={float(flow_time):g} "
        f"from one raw file ({selection_label} of {n_available} available): {path}"
    )
    values = _load_wtemp_series_from_file(
        path,
        flow_time=flow_time,
        l_size=l_size,
        t_size=t_size,
        thermalization_cut=thermalization_cut,
    )
    if not values.size:
        raise ValueError("No W_temp samples remain after the selected thermalization cut in this file.")
    return values, path, selected_index, n_available


def _automatic_tau_window(tau_by_lag, window_c):
    for lag in range(1, tau_by_lag.size):
        if lag >= float(window_c) * tau_by_lag[lag]:
            return lag
    return int(tau_by_lag.size - 1)


def integrated_autocorrelation(series, *, max_lag=None, window_c=5.0, window_m=None):
    values = np.asarray(series, dtype=np.float64)
    values = values[np.isfinite(values)]
    n = int(values.size)
    if n < 2:
        raise ValueError("At least two finite samples are required for autocorrelation.")

    centered = values - np.mean(values, dtype=np.float64)
    variance = float(np.mean(centered * centered, dtype=np.float64))
    if variance <= 0.0:
        raise ValueError("Cannot calculate autocorrelation for a zero-variance series.")

    max_lag = n - 1 if max_lag is None else min(int(max_lag), n - 1)
    if max_lag < 1:
        raise ValueError("max_lag must be at least 1 for tau_int.")

    fft_size = 1 << (2 * n - 1).bit_length()
    transformed = np.fft.rfft(centered, n=fft_size)
    acov = np.fft.irfft(transformed * np.conjugate(transformed), n=fft_size)[: max_lag + 1]
    normalization = variance * np.arange(n, n - max_lag - 1, -1, dtype=np.float64)
    rho = acov / normalization
    rho[0] = 1.0

    tau_by_lag = np.empty(max_lag + 1, dtype=np.float64)
    tau_by_lag[0] = 0.5
    tau_by_lag[1:] = 0.5 + np.cumsum(rho[1:])
    if window_m is None:
        window = _automatic_tau_window(tau_by_lag, window_c)
    else:
        window = int(window_m)
        if window < 0 or window > max_lag:
            raise ValueError(f"window_m must be between 0 and max_lag={max_lag}, got {window}")

    return {
        "n_samples": n,
        "mean": float(np.mean(values, dtype=np.float64)),
        "variance": variance,
        "max_lag": int(max_lag),
        "window_c": float(window_c),
        "window": int(window),
        "tau_int": float(tau_by_lag[window]),
        "rho": rho,
        "tau_by_lag": tau_by_lag,
    }


def calculate_tau_int_for_wtemp_key(
    summary,
    *,
    flow_time,
    wilson_loop,
    thermalization_cut,
    max_lag,
    window_c,
    window_m=None,
    input_path=None,
    input_index=0,
):
    l_size, t_size = wilson_loop
    series, source_path, source_index, n_available = load_wtemp_series_for_key(
        summary,
        flow_time=flow_time,
        l_size=l_size,
        t_size=t_size,
        thermalization_cut=thermalization_cut,
        input_path=input_path,
        input_index=input_index,
    )
    result = integrated_autocorrelation(series, max_lag=max_lag, window_c=window_c, window_m=window_m)
    result.update(
        {
            "flow_time": float(flow_time),
            "wilson_loop": (int(l_size), int(t_size)),
            "thermalization_cut": float(thermalization_cut),
            "source_path": str(source_path),
            "source_index": None if source_index is None else int(source_index),
            "n_available_input_paths": int(n_available),
        }
    )
    return result


def calculate_tau_int_for_all_wtemp_files(summary, *, flow_time, wilson_loop, thermalization_cut, max_lag, window_c, window_m=None):
    paths = _summary_input_paths(summary)
    l_size, t_size = wilson_loop
    results = []
    print(
        f"Calculating W({int(l_size)},{int(t_size)}) tau_int at t/a^2={float(flow_time):g} "
        f"for {len(paths)} independent raw file(s)..."
    )
    for index, path in enumerate(paths):
        print(f"  file {index + 1}/{len(paths)}: {path}")
        series = _load_wtemp_series_from_file(
            path,
            flow_time=flow_time,
            l_size=l_size,
            t_size=t_size,
            thermalization_cut=thermalization_cut,
        )
        if not series.size:
            print("    skipped: no samples remain after the thermalization cut")
            continue
        result = integrated_autocorrelation(series, max_lag=max_lag, window_c=window_c, window_m=window_m)
        result.update(
            {
                "flow_time": float(flow_time),
                "wilson_loop": (int(l_size), int(t_size)),
                "thermalization_cut": float(thermalization_cut),
                "source_path": str(path),
                "source_index": int(index),
                "n_available_input_paths": int(len(paths)),
            }
        )
        results.append(result)
    if not results:
        raise ValueError("No tau_int results could be calculated for the selected files.")
    return results


def select_tau_result(results, *, input_path=None, input_index=0):
    if input_path is not None:
        target = str(Path(input_path).expanduser())
        for result in results:
            if str(Path(result["source_path"]).expanduser()) == target:
                return result
        raise ValueError(f"TAU_INPUT_PATH is not part of the calculated summary input paths: {target}")

    index = int(input_index)
    for result in results:
        if result.get("source_index") == index:
            return result
    raise IndexError(f"TAU_INPUT_INDEX {index} was not calculated.")


def build_tau_int_figure(result, *, width=950, height=560):
    lags = np.arange(result["tau_by_lag"].size)
    window = int(result["window"])
    tau = float(result["tau_int"])
    l_size, t_size = result["wilson_loop"]
    flow_time = float(result["flow_time"])
    cut = float(result["thermalization_cut"])

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=lags,
            y=result["tau_by_lag"],
            mode="lines",
            name="tau_int(M)",
            hovertemplate="M=%{x}<br>tau_int(M)=%{y:.6g}<extra></extra>",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=[window],
            y=[tau],
            mode="markers",
            name=f"window M={window}",
            marker={"size": 9, "color": "red"},
            hovertemplate="M=%{x}<br>tau_int=%{y:.6g}<extra></extra>",
        )
    )
    fig.add_annotation(x=window, y=tau, text=f"tau_int~{int(round(tau))}", showarrow=True, ax=45, ay=-35)
    fig.update_layout(
        title=None,
        width=width,
        height=height,
        xaxis_title="window M",
        yaxis_title="tau_int(M)",
        margin={"l": 56, "r": 12, "t": 12, "b": 46},
    )
    return fig


def build_autocorrelation_figure(result, *, width=950, height=560):
    lags = np.arange(result["rho"].size)
    window = int(result["window"])
    l_size, t_size = result["wilson_loop"]
    flow_time = float(result["flow_time"])
    cut = float(result["thermalization_cut"])

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=lags,
            y=result["rho"],
            mode="lines",
            name="rho(k)",
            hovertemplate="k=%{x}<br>rho(k)=%{y:.6g}<extra></extra>",
        )
    )
    fig.update_layout(
        title=None,
        width=width,
        height=height,
        xaxis_title="lag k",
        yaxis_title="rho(k)",
        margin={"l": 48, "r": 12, "t": 12, "b": 46},
    )
    return fig


def build_all_tau_int_figure(results, *, width=950, height=560, opacity=0.35):
    if not results:
        raise ValueError("No per-file tau_int results are available.")
    first = results[0]
    l_size, t_size = first["wilson_loop"]
    flow_time = float(first["flow_time"])
    cut = float(first["thermalization_cut"])

    fig = go.Figure()
    for result in results:
        lags = np.arange(result["tau_by_lag"].size)
        source_index = result.get("source_index")
        source_name = Path(result["source_path"]).parent.name
        fig.add_trace(
            go.Scatter(
                x=lags,
                y=result["tau_by_lag"],
                mode="lines",
                name=f"file {int(source_index) + 1}: {source_name}" if source_index is not None else source_name,
                opacity=float(opacity),
                customdata=np.column_stack(
                    [
                        np.full(lags.size, int(result["window"])),
                        np.full(lags.size, float(result["tau_int"])),
                        np.full(lags.size, int(result["n_samples"])),
                    ]
                ),
                hovertemplate=(
                    "M=%{x}<br>tau_int(M)=%{y:.6g}<br>"
                    "window=%{customdata[0]}<br>tau_int=%{customdata[1]:.6g}<br>n=%{customdata[2]}<extra>%{fullData.name}</extra>"
                ),
            )
        )

    fig.update_layout(
        title=None,
        width=width,
        height=height,
        xaxis_title="window M",
        yaxis_title="tau_int(M)",
        legend_title="Source file",
        margin={"l": 70, "r": 40, "t": 85, "b": 70},
    )
    return fig


def average_autocorrelation_result(results, *, window_c=5.0, window_m=None):
    if not results:
        raise ValueError("No per-file autocorrelation results are available.")
    common_length = min(result["rho"].size for result in results)
    rho_stack = np.vstack([result["rho"][:common_length] for result in results])
    rho_average = np.mean(rho_stack, axis=0)
    rho_std = np.std(rho_stack, axis=0, ddof=1) if len(results) > 1 else np.zeros(common_length)

    tau_by_lag = np.empty(common_length, dtype=np.float64)
    tau_by_lag[0] = 0.5
    if common_length > 1:
        tau_by_lag[1:] = 0.5 + np.cumsum(rho_average[1:])
    if window_m is None:
        window = _automatic_tau_window(tau_by_lag, window_c)
    else:
        window = int(window_m)
        if window < 0 or window >= common_length:
            raise ValueError(f"window_m must be between 0 and {common_length - 1}, got {window}")

    first = results[0]
    return {
        "flow_time": float(first["flow_time"]),
        "wilson_loop": first["wilson_loop"],
        "thermalization_cut": float(first["thermalization_cut"]),
        "n_files": int(len(results)),
        "n_samples_by_file": [int(result["n_samples"]) for result in results],
        "max_lag": int(common_length - 1),
        "window_c": float(window_c),
        "window": int(window),
        "tau_int": float(tau_by_lag[window]),
        "rho": rho_average,
        "rho_std": rho_std,
        "tau_by_lag": tau_by_lag,
    }


def build_average_autocorrelation_figure(result, *, width=950, height=560):
    lags = np.arange(result["rho"].size)
    l_size, t_size = result["wilson_loop"]
    flow_time = float(result["flow_time"])
    cut = float(result["thermalization_cut"])
    window = int(result["window"])
    tau = float(result["tau_int"])

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=lags,
            y=result["rho"],
            mode="lines",
            name="average rho(k)",
            hovertemplate="k=%{x}<br>average rho(k)=%{y:.6g}<extra></extra>",
        )
    )
    fig.update_layout(
        title=None,
        width=width,
        height=height,
        xaxis_title="lag k",
        yaxis_title="average rho(k)",
        margin={"l": 70, "r": 40, "t": 85, "b": 70},
    )
    return fig


tau_results_by_file = calculate_tau_int_for_all_wtemp_files(
    summary,
    flow_time=TAU_FLOW_TIME,
    wilson_loop=TAU_WILSON_LOOP,
    thermalization_cut=TAU_THERMALIZATION_CUT,
    max_lag=TAU_MAX_LAG,
    window_c=TAU_WINDOW_C,
    window_m=TAU_WINDOW_M,
)
tau_result = select_tau_result(tau_results_by_file, input_path=TAU_INPUT_PATH, input_index=TAU_INPUT_INDEX)
tau_average_autocorrelation = average_autocorrelation_result(tau_results_by_file, window_c=TAU_WINDOW_C, window_m=TAU_WINDOW_M)
fig_tau_int = build_tau_int_figure(tau_result, width=FIGURE_WIDTH, height=FIGURE_HEIGHT)
fig_tau_autocorrelation = build_autocorrelation_figure(tau_result, width=FIGURE_WIDTH, height=FIGURE_HEIGHT)
fig_tau_int_all = build_all_tau_int_figure(
    tau_results_by_file,
    width=FIGURE_WIDTH,
    height=FIGURE_HEIGHT,
    opacity=TAU_ALL_FILES_OPACITY,
)
fig_tau_avg_autocorrelation = build_average_autocorrelation_figure(
    tau_average_autocorrelation,
    width=FIGURE_WIDTH,
    height=FIGURE_HEIGHT,
)
figure_registry["tau_int"] = fig_tau_int
figure_registry["tau_autocorrelation"] = fig_tau_autocorrelation
figure_registry["tau_int_all"] = fig_tau_int_all
figure_registry["tau_avg_autocorrelation"] = fig_tau_avg_autocorrelation
print(
    f"example tau_int W({tau_result['wilson_loop'][0]},{tau_result['wilson_loop'][1]}) "
    f"at t/a^2={tau_result['flow_time']:g} after cut >= {tau_result['thermalization_cut']:g}: "
    f"{tau_result['tau_int']:.12g} (window M={tau_result['window']}, n={tau_result['n_samples']})"
)
print(f"source file: {tau_result['source_path']}")
print(
    f"average-rho tau_int over {tau_average_autocorrelation['n_files']} files: "
    f"{tau_average_autocorrelation['tau_int']:.12g} "
    f"(window M={tau_average_autocorrelation['window']})"
)
if SAVE_TAU_INT_PDF:
    tau_int_pdf_path = (
        Path(TAU_INT_PDF_PATH).expanduser()
        if TAU_INT_PDF_PATH is not None
        else analysis_dir / "tau_int_single_file.pdf"
    )
    tau_int_pdf_path.parent.mkdir(parents=True, exist_ok=True)
    fig_tau_int.write_image(str(tau_int_pdf_path))
    print(f"Saved single-file tau_int PDF: {tau_int_pdf_path}")

if SAVE_TAU_AUTOCORRELATION_PDF:
    tau_autocorrelation_pdf_path = (
        Path(TAU_AUTOCORRELATION_PDF_PATH).expanduser()
        if TAU_AUTOCORRELATION_PDF_PATH is not None
        else analysis_dir / "tau_autocorrelation_single_file.pdf"
    )
    tau_autocorrelation_pdf_path.parent.mkdir(parents=True, exist_ok=True)
    fig_tau_autocorrelation.write_image(str(tau_autocorrelation_pdf_path))
    print(f"Saved single-file autocorrelation PDF: {tau_autocorrelation_pdf_path}")

try:
    from IPython.display import display as _display
except Exception:
    _display = None

if _display is not None:
    _display(fig_tau_int)
    _display(fig_tau_autocorrelation)
    _display(fig_tau_int_all)
    _display(fig_tau_avg_autocorrelation)


Calculating W(4,4) tau_int at t/a^2=1 for 40 independent raw file(s)...
  file 1/40: /home/pietl/masterarbeit/data/20260524/25/gradient_flow_wtemp.dat
  file 2/40: /home/pietl/masterarbeit/data/20260524/26/gradient_flow_wtemp.dat
  file 3/40: /home/pietl/masterarbeit/data/20260524/27/gradient_flow_wtemp.dat
  file 4/40: /home/pietl/masterarbeit/data/20260524/28/gradient_flow_wtemp.dat
  file 5/40: /home/pietl/masterarbeit/data/20260524/29/gradient_flow_wtemp.dat
  file 6/40: /home/pietl/masterarbeit/data/20260524/30/gradient_flow_wtemp.dat
  file 7/40: /home/pietl/masterarbeit/data/20260524/31/gradient_flow_wtemp.dat
  file 8/40: /home/pietl/masterarbeit/data/20260524/32/gradient_flow_wtemp.dat
  file 9/40: /home/pietl/masterarbeit/data/20260531/1/gradient_flow_wtemp.dat
  file 10/40: /home/pietl/masterarbeit/data/20260531/10/gradient_flow_wtemp.dat
  file 11/40: /home/pietl/masterarbeit/data/20260531/11/gradient_flow_wtemp.dat
  file 12/40: /home/pietl/masterarbeit/data/20260531/12/gr

In [9]:
def build_pdf_figure(name):
    if name == "wilson":
        export_flow = SELECTED_FLOW_TIME if SELECTED_FLOW_TIME is not None else flow_times[0]
        return build_wilson_loop_figure(
            records,
            flow_time=export_flow,
            l_values=SELECTED_L_VALUES,
            t_values=SELECTED_T_VALUES,
            show_error_bars=SHOW_ERROR_BARS,
            log_y=LOG_Y,
            width=FIGURE_WIDTH,
            height=FIGURE_HEIGHT,
        )
    if name == "thermalization":
        if thermalization_payload is None:
            raise ValueError("thermalization_preview_data.json is not available for this analysis folder.")
        return build_thermalization_figure(
            thermalization_payload,
            selected_key=THERMALIZATION_KEY,
            max_series_per_key=THERMALIZATION_MAX_SERIES_PER_KEY,
            visible_run_labels=THERMALIZATION_VISIBLE_RUNS,
            x_range=THERMALIZATION_X_RANGE,
            raw_y_range=THERMALIZATION_RAW_Y_RANGE,
            cut=THERMALIZATION_CUT,
            dropdown=THERMALIZATION_SHOW_SELECTOR,
            width=FIGURE_WIDTH,
            height=FIGURE_HEIGHT,
        )
    if name == "block_size":
        if block_size_payload is None:
            raise ValueError("bootstrap_block_size_scan.json is not available for this analysis folder.")
        return build_block_size_figure(
            block_size_payload,
            selected_keys=BLOCK_SIZE_KEYS,
            x_range=BLOCK_SIZE_X_RANGE,
            y_range=BLOCK_SIZE_Y_RANGE,
            default_block_size=BLOCK_SIZE_DEFAULT,
            width=FIGURE_WIDTH,
            height=FIGURE_HEIGHT,
        )
    if name == "tau_int":
        if "tau_result" not in globals():
            raise ValueError("Run the integrated autocorrelation cell before exporting PDF_FIGURE='tau_int'.")
        return build_tau_int_figure(tau_result, width=FIGURE_WIDTH, height=FIGURE_HEIGHT)
    if name == "tau_autocorrelation":
        if "tau_result" not in globals():
            raise ValueError("Run the integrated autocorrelation cell before exporting PDF_FIGURE='tau_autocorrelation'.")
        return build_autocorrelation_figure(tau_result, width=FIGURE_WIDTH, height=FIGURE_HEIGHT)
    if name == "tau_int_all":
        if "tau_results_by_file" not in globals():
            raise ValueError("Run the integrated autocorrelation cell before exporting PDF_FIGURE='tau_int_all'.")
        return build_all_tau_int_figure(
            tau_results_by_file,
            width=FIGURE_WIDTH,
            height=FIGURE_HEIGHT,
            opacity=TAU_ALL_FILES_OPACITY,
        )
    if name == "tau_avg_autocorrelation":
        if "tau_average_autocorrelation" not in globals():
            raise ValueError("Run the integrated autocorrelation cell before exporting PDF_FIGURE='tau_avg_autocorrelation'.")
        return build_average_autocorrelation_figure(
            tau_average_autocorrelation,
            width=FIGURE_WIDTH,
            height=FIGURE_HEIGHT,
        )
    raise ValueError(f"Unknown PDF_FIGURE: {name!r}")


pdf_path = Path(PDF_PATH).expanduser() if PDF_PATH is not None else analysis_dir / f"{PDF_FIGURE}_plot.pdf"
pdf_path.parent.mkdir(parents=True, exist_ok=True)
fig_pdf = build_pdf_figure(PDF_FIGURE)
fig_pdf.write_image(str(pdf_path))
print(f"Saved PDF: {pdf_path}")


Saved PDF: /home/pietl/masterarbeit/data/gradient_flow_wtemp_analysis/beta_2p4__L0_24__epsilon1_0__dt_0p01__b6c467be/wilson_plot.pdf
